# FireGuard AI — Full GPU Model Training (Google Colab)

This notebook downloads the **complete official D-Fire YOLO dataset (~21,527 images: 15,499 train / 1,722 valid / 4,306 test)**, validates labels, trains **YOLO11n** for 80 epochs on a Colab GPU, evaluates on the held-out test split, and downloads your production `best.pt` weights.

> **Before Running:** In Google Colab, select **Runtime -> Change runtime type -> T4 GPU** (or A100/L4).

In [ ]:
# Cell 1: Detect GPU availability
import platform
import torch

print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    !nvidia-smi
else:
    print("WARNING: No GPU detected. Switch runtime to T4 GPU before training all 21,527 images.")

In [ ]:
# Cell 2: Install compatible dependencies
!pip install -q "ultralytics>=8.3.0,<9.0.0" "PyYAML>=6.0.1" requests scikit-learn pandas matplotlib seaborn tqdm

In [ ]:
# Cell 3: Configure workspace (Google Drive or Colab ephemeral storage)
import os
from pathlib import Path

USE_GOOGLE_DRIVE = False  # Set True if you uploaded your FireGuard-AI project to Google Drive
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/FireGuard-AI')
else:
    PROJECT_DIR = Path('/content/FireGuard-AI')

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)
print(f"Working directory: {Path.cwd()}")

In [ ]:
# Cell 4: Download & extract full official D-Fire YOLO dataset (21,527 images, classes 0=smoke, 1=fire)
import shutil
import zipfile
from pathlib import Path
import yaml
from tqdm import tqdm
import requests

DFIRE_ZIP_URL = "https://huggingface.co/datasets/Stevebirth/fire-ulm-dfire/resolve/main/dfire_yolo_20261016.zip"
RAW_ZIP = Path('/content/dfire_yolo_20261016.zip')
DATASET_DIR = Path('dataset')

if not RAW_ZIP.is_file():
    print('Downloading complete D-Fire YOLO dataset archive...')
    with requests.get(DFIRE_ZIP_URL, stream=True, timeout=120) as resp:
        resp.raise_for_status()
        total = int(resp.headers.get('content-length', 0))
        with open(RAW_ZIP, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True) as bar:
            for chunk in resp.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))

print('Extracting official train / val / test splits...')
extract_root = Path('/content/dfire_extracted')
if not (extract_root / 'dfire_yolo').is_dir():
    with zipfile.ZipFile(RAW_ZIP, 'r') as zf:
        zf.extractall(extract_root)

src_root = extract_root / 'dfire_yolo'
split_map = {'train': 'train', 'val': 'valid', 'test': 'test'}
for src_split, dst_split in split_map.items():
    for sub in ('images', 'labels'):
        dst_dir = DATASET_DIR / dst_split / sub
        if dst_dir.exists():
            shutil.rmtree(dst_dir)
        shutil.copytree(src_root / src_split / sub, dst_dir)

# Optional: Clone DFireDataset GitHub repository for dataset attribution & license notice
if not Path('/content/DFireDataset_repo').is_dir():
    !git clone --depth 1 https://github.com/gaia-solutions-on-demand/DFireDataset.git /content/DFireDataset_repo

# Optional negative-only images folder (e.g., extracted not_fire from D:\archive.zip)
# If you upload hard negative normal images into /content/extra_not_fire, this creates empty .txt labels
EXTRA_NEG_DIR = Path('/content/extra_not_fire')
if EXTRA_NEG_DIR.is_dir():
    added = 0
    for img_p in EXTRA_NEG_DIR.rglob('*'):
        if img_p.suffix.lower() in {'.jpg', '.jpeg', '.png'} and added < 2000:
            out_name = f'extra_neg_{added}_{img_p.name}'
            shutil.copy2(img_p, DATASET_DIR / 'train' / 'images' / out_name)
            (DATASET_DIR / 'train' / 'labels' / f'{Path(out_name).stem}.txt').write_text('', encoding='utf-8')
            added += 1
    print(f'Added {added} extra negative background images to train split.')

data_yaml = {
    'path': str(DATASET_DIR.resolve()),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 2,
    'names': {0: 'smoke', 1: 'fire'},
}
(DATASET_DIR / 'data.yaml').write_text(yaml.safe_dump(data_yaml, sort_keys=False), encoding='utf-8')
print('Wrote dataset/data.yaml:', (DATASET_DIR / 'data.yaml').resolve())

In [ ]:
# Cell 5: Audit dataset splits, image counts, and label box class distribution
from collections import Counter

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
for sp in ('train', 'valid', 'test'):
    imgs = [p for p in (DATASET_DIR / sp / 'images').iterdir() if p.suffix.lower() in IMG_EXTS]
    cls_counts = Counter()
    neg_imgs = 0
    for img_p in imgs:
        lbl_p = DATASET_DIR / sp / 'labels' / f'{img_p.stem}.txt'
        lines = [ln.strip() for ln in lbl_p.read_text(encoding='utf-8').splitlines() if ln.strip()] if lbl_p.exists() else []
        if not lines:
            neg_imgs += 1
        for ln in lines:
            cls_id = int(float(ln.split()[0]))
            cls_counts[cls_id] += 1
    print(f"{sp:5s}: {len(imgs):5d} images | empty/negative={neg_imgs:4d} | smoke_boxes(0)={cls_counts[0]:5d} | fire_boxes(1)={cls_counts[1]:5d}")

In [ ]:
# Cell 6: Train YOLO11n on full D-Fire dataset (80 epochs, imgsz=640, cos_lr=True)
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 16 if torch.cuda.is_available() else 8

model = YOLO('yolo11n.pt')  # Pretrained COCO backbone for transfer learning
train_results = model.train(
    data=str((DATASET_DIR / 'data.yaml').resolve()),
    epochs=80,
    imgsz=640,
    batch=BATCH_SIZE,
    optimizer='auto',
    patience=15,
    cos_lr=True,
    seed=42,
    device=DEVICE,
    val=True,
    save=True,
    plots=True,
    project='runs/detect',
    name='fireguard_yolo11n_full',
    exist_ok=True,
)
print(f"Training finished. Outputs saved in: {train_results.save_dir}")

In [ ]:
# Cell 7: Plot real training curves & confusion matrix
from IPython.display import Image as IPyImage, display
import pandas as pd
import matplotlib.pyplot as plt

run_dir = Path('runs/detect/fireguard_yolo11n_full')
results_csv = run_dir / 'results.csv'
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    display(df.tail(5))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for col in ['train/box_loss', 'val/box_loss', 'train/cls_loss', 'val/cls_loss']:
        if col in df.columns:
            axes[0].plot(df['epoch'], df[col], label=col)
    axes[0].set_title('Loss Curves'); axes[0].legend()

    for col in ['metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)']:
        if col in df.columns:
            axes[1].plot(df['epoch'], df[col], label=col)
    axes[1].set_title('Validation Metrics'); axes[1].legend()
    plt.show()

for img_name in ['confusion_matrix_normalized.png', 'BoxPR_curve.png']:
    p = run_dir / img_name
    if p.exists():
        display(IPyImage(filename=str(p)))

In [ ]:
# Cell 8: Evaluate best.pt on the independent held-out D-Fire test split
best_weights = run_dir / 'weights' / 'best.pt'
eval_model = YOLO(str(best_weights))
test_res = eval_model.val(
    data=str((DATASET_DIR / 'data.yaml').resolve()),
    split='test',
    imgsz=640,
    conf=0.25,
    iou=0.45,
    plots=True,
)
print('=== Held-Out Test Split Results ===')
print(f'Precision : {test_res.box.mp:.4f}')
print(f'Recall    : {test_res.box.mr:.4f}')
print(f'mAP@50    : {test_res.box.map50:.4f}')
print(f'mAP@50-95 : {test_res.box.map:.4f}')
for idx, name in sorted(eval_model.names.items()):
    print(f'  Class {idx} ({name}): P={test_res.box.p[idx]:.4f}, R={test_res.box.r[idx]:.4f}, mAP50={test_res.box.ap50[idx]:.4f}')

In [ ]:
# Cell 9: Package models/best.pt + model_metadata.json and download to your PC
import json

Path('models').mkdir(exist_ok=True)
shutil.copy2(best_weights, 'models/best.pt')

metadata = {
    'model_tier': 'FULL_EVALUATED',
    'base_model': 'yolo11n.pt',
    'epochs': 80,
    'imgsz': 640,
    'device': str(DEVICE),
    'classes': {'0': 'smoke', '1': 'fire'},
    'test_metrics': {
        'precision': round(float(test_res.box.mp), 4),
        'recall': round(float(test_res.box.mr), 4),
        'map50': round(float(test_res.box.map50), 4),
        'map50_95': round(float(test_res.box.map), 4),
    },
    'note': 'Full YOLO11n model trained on complete D-Fire dataset in Google Colab GPU.',
}
Path('models/model_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')

try:
    from google.colab import files
    files.download('models/best.pt')
    files.download('models/model_metadata.json')
except ImportError:
    print('Saved locally to models/best.pt and models/model_metadata.json')

### Placing Colab Weights Locally
1. Download `best.pt` and `model_metadata.json` from Cell 9.
2. Replace `c:\Users\kakum\Desktop\fire\models\best.pt` and `c:\Users\kakum\Desktop\fire\models\model_metadata.json` (keep `models/smoke_test_best.pt` as backup if desired).
3. Restart Streamlit (`streamlit run app.py`) and test on your images and videos!